In [3]:
import gcsfs
import pandas as pd
import os
from _utils import GCS_FILE_PATH
import geopandas as gpd
import folium
import altair as alt

import matplotlib.pyplot as plt
from pywaffle import Waffle
from pypalettes import load_palette
from pyfonts import load_google_font

In [85]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual_service_and_opex.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [5]:
df.head(2)

,key,ntd_id,mode,year,type_of_service,unlinked_passenger_trips,vehicle_revenue_hours,vehicle_revenue_miles,vehicles_operated_in_maxiumum_service,passenger_miles_traveled,...,uza_population,source_agency,source_city,source_state,upt_prior_year,upt_change_1yr,upt_pct_change_1yr,mode_full_name,type_of_service_full_name,service_type
0,8ba07c3a4edfcb413278c890a0b3b541,00007,MB,2024,PT,13865,5784,129301,3,356741,...,270179,Lane Transit District (LTD),Eugene,OR,13050,815,0.0625,Bus,Purchased Transportation,Fixed Route
1,768d38a1c4079ce315353b735ec41298,00007,VP,2020,PT,30475,6810,312462,16,1661859,...,270179,Lane Transit District (LTD),Eugene,OR,40994,-10519,-0.2566,Vanpool,Purchased Transportation,Demand Response


Waffle Chart for time series 

In [6]:
df.columns

Index(['key', 'ntd_id', 'mode', 'year', 'type_of_service',
       'unlinked_passenger_trips', 'vehicle_revenue_hours',
       'vehicle_revenue_miles', 'vehicles_operated_in_maxiumum_service',
       'passenger_miles_traveled', 'direction_route_miles',
       'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'operating_expenses_total',
       'fare_revenue', 'opex_per_vrh', 'opex_per_vrm', 'opex_per_upt',
       'upt_per_vrh', 'upt_per_vrm', 'farebox_recovery_ratio', 'agency_status',
       'census_year', 'last_report_year', 'mode_status', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles',
       'primary_uza_name', 'uza_population', 'source_agency', 'source_city',
       'source_state', 'upt_prior_year', 'upt_change_1yr',
       'upt_pct_change_1yr', 'mode_full_name', 'type_of_service_full_name',
       'service_type'

In [7]:
df_agg =(df
    .groupby(["year", "mode"])
    ["unlinked_passenger_trips"]
    .sum()
    .reset_index())

In [8]:
all_years = df_agg["year"].unique()
all_modes = df_agg["mode"].unique()

idx = pd.MultiIndex.from_product(
    [all_years, all_modes], names=["year", "mode"]
)

df_agg = (
    df_agg.set_index(["year", "mode"])
    .reindex(idx, fill_value=0)
    .reset_index()
)

df_agg = df_agg.sort_values(["year", "mode"])
df_agg[df_agg["year"] == 2024].sort_values("unlinked_passenger_trips")
df_agg["israilbus"] = df_agg["mode"].isin(["MB", "DR"])
df_agg.head()

,year,mode,unlinked_passenger_trips,israilbus
0,2015,CB,12060734,False
1,2015,CC,6834184,False
2,2015,CR,35821800,False
3,2015,DR,23795831,True
4,2015,FB,5031924,False


In [86]:
df_CA = df[df["primary_uza_name"].str.endswith(", CA", na=False)]

In [39]:
top_agencies = (
    df_CA[df_CA["year"] == 2024].
        groupby("source_agency")["unlinked_passenger_trips"].sum().nlargest(10).index.tolist()
)

agency_df = df_CA[(df_CA["source_agency"].isin(top_agencies)) & (df_CA["year"] == 2024)].copy()

expense_cols = {"Vehicle Operations":"operating_expenses_vehicle_operations",
                "Vehicle Maintenance": "operating_expenses_vehicle_maintenance",
                "Non-Vehicle Maintenance": "operating_expenses_nonvehicle_maintenance",
                "General Administration": "operating_expenses_general_administration",
}

expenses = (agency_df.groupby("source_agency")[list(expense_cols.values())+["operating_expenses_total"]].sum().reset_index())

                

In [40]:
waterfall = (expenses[
        ["source_agency"] + list(expense_cols.values())].melt(
        id_vars="source_agency",
        var_name="expense_column",
        value_name="amount"
    )
)

waterfall["expense"] = waterfall["expense_column"].map(
    {v: k for k, v in expense_cols.items()}
)


waterfall = waterfall.merge(
    expenses[["source_agency", "operating_expenses_total"]],
    on="source_agency"
)

waterfall["percentage"] = (waterfall["amount"] / waterfall["operating_expenses_total"] * 100)


In [41]:
chart = (
    alt.Chart(waterfall)
    .mark_bar()
    .encode(
        x=alt.X("source_agency:N", title="Agency", sort=top_agencies,
            axis=alt.Axis(labelAngle=-25)),
        y=alt.Y("sum(percentage):Q", title="Operating Expenses (%)",
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(format=".0f")),
        color=alt.Color("expense:N", title="Expense Category"),
        tooltip=[alt.Tooltip("source_agency:N", title="Agency"),
                 alt.Tooltip("expense:N", title="Expense Category"),
                 alt.Tooltip("percentage:Q", title="Share", format=".1f"),
                 alt.Tooltip("amount:Q", title="Amount", format="$,.0f")]
    )
    .properties(title=alt.TitleParams(
        text="Operating Expense Composition — Top 10 Agencies",
        subtitle=("2024 | Ranked by unlinked passenger trips | "
                "Each bar represents 100% of operating expenses")),
        width=750,
        height=450)
)

chart

alt.Chart(...)

In [82]:
import altair as alt

# Create ONE observation per agency + mode for 2024
ridge_df = (
    df_CA[(df_CA["year"] == 2024) & (df_CA["mode_full_name"] != "Vanpool")]
    .groupby(["source_agency", "mode_full_name"], as_index=False
    )
    .agg(fare_revenue=("fare_revenue", "sum"), operating_expenses_total=("operating_expenses_total", "sum"))
)

# Calculate agency-level farebox recovery ratio
ridge_df["farebox_recovery_ratio"] = (ridge_df["fare_revenue"] /ridge_df["operating_expenses_total"])

# Remove invalid values
ridge_df = ridge_df[ridge_df["farebox_recovery_ratio"].notna() & (ridge_df["farebox_recovery_ratio"] >= 0) & (ridge_df["farebox_recovery_ratio"] <= 2)].copy()

step = 30
overlap = 1.5

chart = (
    alt.Chart(ridge_df, width=800, height=step)
    .transform_density(
        "farebox_recovery_ratio",
        groupby=["mode_full_name"],
        as_=["farebox_recovery_ratio", "density"]
    )
    .mark_area(
        interpolate="monotone",
        fillOpacity=0.8,
        stroke="white",
        strokeWidth=0.5
    )
    .encode(
        x=alt.X("farebox_recovery_ratio:Q", title="Farebox Recovery Ratio",
            axis=alt.Axis(format=".0%")),
        y=alt.Y("density:Q",
            axis=None, scale=alt.Scale(range=[step, -step * overlap])),
        color=alt.Color("mode_full_name:N", title="Mode"),
        tooltip=[alt.Tooltip("mode_full_name:N",title="Mode"),
                 alt.Tooltip("farebox_recovery_ratio:Q", title="Farebox Recovery Ratio", format=".1%"),
                 alt.Tooltip("density:Q", title="Density", format=".3f")]
    )
    .facet(row=alt.Row("mode_full_name:N", title=None,
            header=alt.Header(labelAngle=0, labelAlign="left"))
    )
    .properties(title=alt.TitleParams(
            text="Farebox Recovery Ratio Distribution by Transit Mode",
            subtitle="2024 | One observation per California agency per mode"),
                bounds="flush")
    .configure_facet(spacing=0)
        .configure_view(stroke=None))

chart

alt.FacetChart(...)

In [104]:

urban = df_CA[
    (df_CA["year"] == 2024) &
    (df_CA["reporting_module"] == "Urban") &
    (df_CA["upt_pct_change_1yr"].notna())
].copy()

# Top 10 increases and top 10 decreases
top_gainers = urban.nlargest(10, "upt_pct_change_1yr")
top_losers = urban.nsmallest(10, "upt_pct_change_1yr")

plot_df = pd.concat([top_losers, top_gainers])

chart = (
    alt.Chart(plot_df)
    .mark_bar()
    .encode(
        y=alt.Y(
            "source_agency:N",
            sort=alt.SortField(
                field="upt_pct_change_1yr",
                order="ascending"
            ),
            title="Agency"
        ),
        x=alt.X(
            "upt_pct_change_1yr:Q",
            title="Year-over-Year Change in UPT",
            axis=alt.Axis(format=".0%")
        ),
        color=alt.Color(
            "type_of_service_full_name:N",
            title="Type of Service"
        ),
        tooltip=[
            alt.Tooltip(
                "source_agency:N",
                title="Agency"
            ),
            alt.Tooltip(
                "type_of_service_full_name:N",
                title="Type of Service"
            ),
            alt.Tooltip(
                "upt_pct_change_1yr:Q",
                title="UPT Change",
                format=".1%"
            ),
            alt.Tooltip(
                "unlinked_passenger_trips:Q",
                title="2024 UPT",
                format=",.0f"
            )
        ]
    )
    .properties(
        title=alt.TitleParams(
            text="Who Grew, Who Shrank? — Urban Transit",
            subtitle=(
                "2024 | Top 10 increases and decreases in "
                "unlinked passenger trips by service type"
            )
        ),
        width=800,
        height=550
    )
)

chart

alt.Chart(...)